# ARCADE Stenosis — **repeat run with a new training seed** (B or N)

**Why:** each 5-fold ensemble was trained once. The two YOLOv8s-seg runs differed by 0.026 in official-style F1, about the size of the N − B gain. This notebook retrains **one** ensemble (B = plain DeepLabV3+ or N = VG-DeepLab) with a **new training seed**, on exactly the same five folds, recipe, post-processing search and test set, and compares it with the original B and N.

**Plan (4 runs, one per Kaggle session):** `WHICH="B", SEED_RUN=1` (~2.6 h) · `WHICH="B", SEED_RUN=2` (~2.6 h) · `WHICH="N", SEED_RUN=1` (~4.3 h) · `WHICH="N", SEED_RUN=2` (~4.3 h). About 14 GPU hours in total, inside the weekly Kaggle quota.

Only the training seed changes (weight initialisation of the new layers, batch order and augmentation draws). The fold split stays `random_state=42`, so OOF images are the same.

## How to run (Kaggle)
1. **Add Input:** the ARCADE dataset (`nirmalgaud/arcade-dataset`) and the output of the first notebook "ARCADE Stenosis — VG-DeepLab" (`dl3_r101_f*.pt`, `vg_dl3_f*.pt`).
2. Accelerator = GPU T4, Internet = On.
3. In the CONFIG cell set `WHICH` and `SEED_RUN`. **Save Version → Save & Run All (Commit)**.
4. Download `results_seed_*.zip` from the Output tab and send it to Claude. After all four runs, Claude combines them (mean and spread over three seeds).


In [1]:
# Do NOT let pip touch torch: --no-deps keeps the preinstalled torch/numpy/opencv exactly as they are.
!pip install -q kagglehub
!pip install -q --no-deps segmentation-models-pytorch
print("Installs complete. If you ever see 'Trying to override a python impl...' -> Run > Restart session, then run from the top (do not re-run this cell).")

Installs complete. If you ever see 'Trying to override a python impl...' -> Run > Restart session, then run from the top (do not re-run this cell).


In [2]:
import os, sys, json, math, copy, random, itertools, shutil, time, warnings, subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
for _try in range(8):          # smp needs a few small packages; install any that are missing WITHOUT touching torch
    try:
        import segmentation_models_pytorch as smp; break
    except ModuleNotFoundError as e:
        print("missing package:", e.name, "-> installing (no deps)")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", e.name.split(".")[0].replace("_", "-")])
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
SEED = 42
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
ON_KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if ON_KAGGLE else (Path("/content") if Path("/content").exists() else Path.cwd())   # works on Kaggle, Colab or a local machine
print("Platform:", "Kaggle" if ON_KAGGLE else ("Colab" if Path("/content").exists() else "local"), "| work dir:", BASE)
print("Torch", torch.__version__, "| smp", smp.__version__, "|", torch.cuda.get_device_name(0) if USE_CUDA else "NO GPU (enable a GPU in the notebook settings!)")

Platform: Kaggle | work dir: /kaggle/working
Torch 2.11.0+cu130 | smp 0.6.0 | Tesla T4


In [3]:
# ============================================================
# CONFIG
# ============================================================
SMOKE_TEST = False         # False = the real run. True = 15-minute plumbing check (1 epoch, ~120 images; numbers mean nothing).
MODE = "best"              # "best": 5 folds x 40 epochs | "fast": 5 folds x 25 epochs (quicker, slightly weaker)
ARCHS = ["dl3_r101"]       # backbone family (supervisor's model)

PRESETS = {"best": dict(N_FOLDS=5, EPOCHS=40), "fast": dict(N_FOLDS=5, EPOCHS=25)}
N_FOLDS, EPOCHS = PRESETS[MODE]["N_FOLDS"], PRESETS[MODE]["EPOCHS"]
BATCH_SIZE, WARMUP_EPOCHS, EMA_DECAY = 8, 2, 0.997
LR_ENCODER, LR_DECODER = 1e-4, 1e-3
POS_WEIGHT, TVERSKY_ALPHA, TVERSKY_BETA = 3.0, 0.5, 0.5
IMG_SIZE, STENOSIS_CATEGORY_ID = 512, 26
N_BOOT = 1000

# ---- ablation of VG-DeepLab ----
WHICH = "B"        # "B" = plain DeepLabV3+ baseline | "N" = full VG-DeepLab
SEED_RUN = 1       # 1 or 2 (the original run is seed run 0, seed 42)
TRAIN_SEED = 42 + 1000 * SEED_RUN
USE_AUX, USE_REFINE = (True, True) if WHICH == "N" else (False, False)
ABLATION = f"seed{SEED_RUN}_{WHICH}"
SYN_BATCH, SYN_EVERY = 8, 2            # identical to the first run
LAMBDA_VESSEL, DS_WEIGHT = 1.0, 0.4    # identical to the first run
REQUIRE_FIRST_RUN = True               # stop at once if the checkpoints of the first run are not attached (instead of silently re-training for hours)
TRAIN_BASELINE_IF_MISSING = True       # only used when REQUIRE_FIRST_RUN = False: train the baseline here (+~2.6 h)

if SMOKE_TEST:
    N_FOLDS, EPOCHS, N_BOOT = 2, 1, 100
    SMOKE_POOL, SMOKE_TEST_N, SMOKE_SYN = 120, 60, 60

RUN_DIR = BASE / ("run_smoke" if SMOKE_TEST else "run"); RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = BASE / ("outputs_smoke" if SMOKE_TEST else "outputs"); OUT_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_ERRORS = []
print(f"SMOKE_TEST={SMOKE_TEST} | ABLATION={ABLATION} (vessel head={USE_AUX}, refinement={USE_REFINE}) | MODE={MODE} | folds={N_FOLDS} | epochs={EPOCHS}")

SMOKE_TEST=False | ABLATION=seed1_B (vessel head=False, refinement=False) | MODE=best | folds=5 | epochs=40


In [4]:
def find_arcade_root():
    """Use an attached Kaggle dataset (/kaggle/input) or an earlier kagglehub download if present."""
    for root in [Path("/kaggle/input"), Path.home() / ".cache" / "kagglehub"]:
        if root.exists():
            c = [p for p in root.rglob("train.json") if "stenosis" in p.parent.parent.parent.name.lower()]
            if c: return c[0].parent.parent.parent
    return None

DATASET_ROOT = find_arcade_root()
if DATASET_ROOT is None:                      # not attached -> download (needs Internet ON)
    import kagglehub
    try: download_path = Path(kagglehub.dataset_download("nirmalgaud/arcade-dataset"))
    except Exception as e: raise RuntimeError("ARCADE is not attached. On Kaggle: right panel -> Add Input -> search 'arcade-dataset' (nirmalgaud) -> Add, then run again.") from e
    cands = [c for c in download_path.rglob("train.json") if "stenosis" in str(c).lower()]
    if not cands: raise FileNotFoundError("No 'stenosis' train.json found (wrong ARCADE sub-task folder).")
    DATASET_ROOT = cands[0].parent.parent.parent
for s in ["train", "val", "test"]:
    assert (DATASET_ROOT / s / "annotations" / f"{s}.json").exists() and (DATASET_ROOT / s / "images").exists()
print("ARCADE root:", DATASET_ROOT)

ARCADE root: /kaggle/input/datasets/nirmalgaud/arcade-dataset/arcade/stenosis


## Official-protocol metric code (copied from your YOLOv8s-seg notebook so numbers are directly comparable)
Instance = one lesion mask. **Official ARCADE mean F1** (`P1`): per image, predicted and GT lesions are matched greedily by Dice; a matched pair scores its Dice; every unmatched prediction or GT lesion scores 0; averaged per image, then over images.

In [5]:
# ============================================================
# Instances + official metrics (identical logic to the YOLOv8s-seg notebook; adds per-image Dice output)
# ============================================================
class Inst:
    """One instance mask, cropped to its bounding box."""
    __slots__ = ("y0", "x0", "m", "area", "score", "extra")
    def __init__(self, y0, x0, m, score=1.0, extra=None):
        self.y0, self.x0, self.m = int(y0), int(x0), m
        self.area = int(m.sum()); self.score, self.extra = float(score), extra
    @property
    def y1(self): return self.y0 + self.m.shape[0]
    @property
    def x1(self): return self.x0 + self.m.shape[1]

def inst_from_full(full_bool, score=1.0):
    ys, xs = np.nonzero(full_bool)
    if len(ys) == 0: return None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    return Inst(y0, x0, full_bool[y0:y1, x0:x1].copy(), score)

def inst_inter(a, b):
    y0, y1 = max(a.y0, b.y0), min(a.y1, b.y1); x0, x1 = max(a.x0, b.x0), min(a.x1, b.x1)
    if y0 >= y1 or x0 >= x1: return 0
    return int(np.count_nonzero(a.m[y0 - a.y0:y1 - a.y0, x0 - a.x0:x1 - a.x0] & b.m[y0 - b.y0:y1 - b.y0, x0 - b.x0:x1 - b.x0]))

def inst_dice(a, b):
    s = a.area + b.area
    return 0.0 if s == 0 else 2.0 * inst_inter(a, b) / s

def inst_iou(a, b):
    i = inst_inter(a, b); u = a.area + b.area - i
    return 0.0 if u == 0 else i / u

def paint(insts, shape):
    canvas = np.zeros(shape, bool)
    for i in insts: canvas[i.y0:i.y1, i.x0:i.x1] |= i.m
    return canvas

def inst_greedy_match(preds, gts, criterion="dice"):
    fn = inst_dice if criterion == "dice" else inst_iou
    pairs = []
    for i, p in enumerate(preds):
        for j, g in enumerate(gts):
            s = fn(p, g)
            if s > 0: pairs.append((s, i, j))
    pairs.sort(key=lambda t: -t[0])
    up, ug, out = set(range(len(preds))), set(range(len(gts))), []
    for s, i, j in pairs:
        if i in up and j in ug: out.append((i, j, s)); up.discard(i); ug.discard(j)
    return out, len(up), len(ug)

def p1_image(preds, gts):
    if not preds and not gts: return 1.0
    pairs, n_up, n_ug = inst_greedy_match(preds, gts, "dice")
    return float(np.mean([s for _, _, s in pairs] + [0.0] * (n_up + n_ug)))

def full_metrics(preds_by_img, items):
    p1, p1s, dimg, iimg = [], [], [], []
    TP = FP = FN = pTP = pFP = pFN = 0
    for k, it in enumerate(items):
        preds, gts = preds_by_img[k], it["gts"]
        p1.append(p1_image(preds, gts))
        pairs, n_up, n_ug = inst_greedy_match(preds, gts, "iou")
        sc, tp = [], 0
        for i, j, s in pairs:
            if s >= 0.5: sc.append(inst_dice(preds[i], gts[j])); tp += 1
            else: sc += [0.0, 0.0]
        sc += [0.0] * (n_up + n_ug)
        p1s.append(float(np.mean(sc)) if sc else 1.0)
        TP += tp; FP += len(preds) - tp; FN += len(gts) - tp
        pr, gt = paint(preds, it["shape"]), paint(gts, it["shape"])
        a, b, c = np.count_nonzero(pr & gt), np.count_nonzero(pr & ~gt), np.count_nonzero(~pr & gt)
        pTP += a; pFP += b; pFN += c
        dimg.append(1.0 if a + b + c == 0 else 2 * a / (2 * a + b + c))
        iimg.append(1.0 if a + b + c == 0 else a / (a + b + c))
    P, R = TP / max(TP + FP, 1e-9), TP / max(TP + FN, 1e-9)
    return {"P1": float(np.mean(p1)), "P1_strict": float(np.mean(p1s)),
            "F1": 2 * P * R / max(P + R, 1e-9), "Precision": P, "Recall": R, "TP": TP, "FP": FP, "FN": FN,
            "Dice": 2 * pTP / max(2 * pTP + pFP + pFN, 1e-9), "IoU": pTP / max(pTP + pFP + pFN, 1e-9),
            "Dice_img": float(np.mean(dimg)), "Dice_img_std": float(np.std(dimg)),
            "IoU_img": float(np.mean(iimg)), "pred_per_img": (TP + FP) / len(items),
            "per_image_P1": np.array(p1), "per_image_Dice": np.array(dimg)}

def mask_map(ranked_by_img, items, thrs=np.arange(0.50, 0.96, 0.05)):
    """COCO-style mask AP (101-point). Returns (AP50, AP50-95)."""
    ious = [np.array([[inst_iou(p, g) for g in it["gts"]] for p in ranked_by_img[k]], np.float32)
            .reshape(len(ranked_by_img[k]), len(it["gts"])) for k, it in enumerate(items)]
    n_gt = sum(len(it["gts"]) for it in items)
    order = sorted(((p.score, k, i) for k, ps in enumerate(ranked_by_img) for i, p in enumerate(ps)), key=lambda d: -d[0])
    aps = []
    for thr in thrs:
        used = [np.zeros(len(it["gts"]), bool) for it in items]; tp = np.zeros(len(order))
        for n, (_, k, i) in enumerate(order):
            if ious[k].shape[1] == 0: continue
            cand = np.where(~used[k], ious[k][i], -1.0); j = int(cand.argmax())
            if cand[j] >= thr: used[k][j] = True; tp[n] = 1
        ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp); rec = ctp / max(n_gt, 1)
        prec = np.maximum.accumulate((ctp / np.maximum(ctp + cfp, 1e-9))[::-1])[::-1] if len(order) else np.zeros(0)
        aps.append(float(np.mean([prec[rec >= r].max() if np.any(rec >= r) else 0.0 for r in np.linspace(0, 1, 101)])))
    return aps[0], float(np.mean(aps))

_g = [Inst(0, 0, np.ones((5, 5), bool))]
print("self-check official F1: perfect =", p1_image(_g, _g), "| one extra false positive =", round(p1_image(_g + [Inst(9, 9, np.ones((3, 3), bool))], _g), 3))

self-check official F1: perfect = 1.0 | one extra false positive = 0.5


In [6]:
# ============================================================
# Load ARCADE: arrays (gray 512x512) + GT instances from the COCO polygons (native resolution)
# ============================================================
def polys_to_mask(segs, h, w):
    full = np.zeros((h, w), np.uint8)
    for poly in segs or []:
        if isinstance(poly, list) and len(poly) >= 6:
            cv2.fillPoly(full, [np.round(np.array(poly, np.float32).reshape(-1, 2)).astype(np.int32)], 1)
    return full

def load_split(split):
    coco = json.load(open(DATASET_ROOT / split / "annotations" / f"{split}.json"))
    anns = {}
    for a in coco["annotations"]:
        if a["category_id"] == STENOSIS_CATEGORY_ID: anns.setdefault(a["image_id"], []).append(a)
    imgs, masks, items = [], [], []
    for im in tqdm(sorted(coco["images"], key=lambda d: d["id"]), desc=f"ARCADE {split}"):
        g = cv2.imread(str(DATASET_ROOT / split / "images" / im["file_name"]), cv2.IMREAD_GRAYSCALE)
        assert g.shape == (IMG_SIZE, IMG_SIZE), f"{im['file_name']} is {g.shape}, expected 512x512 (ARCADE native)"
        h, w = g.shape
        gts, union = [], np.zeros((h, w), np.uint8)
        for a in anns.get(im["id"], []):
            m = polys_to_mask(a.get("segmentation"), h, w); union |= m
            gi = inst_from_full(m.astype(bool))
            if gi is not None: gts.append(gi)
        imgs.append(g); masks.append(union); items.append({"file": im["file_name"], "shape": (h, w), "gts": gts})
    return np.stack(imgs), np.stack(masks), items

ARC = {s: load_split(s) for s in ["train", "val", "test"]}
for s, (a, m, it) in ARC.items():
    n = sum(len(i["gts"]) for i in it)
    print(f"{s:5s}: {len(a)} images | {n} lesions | {n / len(a):.2f} per image")

# pool = train + val (used for cross-validation); test is kept untouched until the very end
POOL_IMGS = np.concatenate([ARC["train"][0], ARC["val"][0]])
POOL_MASKS = np.concatenate([ARC["train"][1], ARC["val"][1]])
POOL_ITEMS = ARC["train"][2] + ARC["val"][2]
TEST_IMGS, TEST_MASKS, TEST_ITEMS = ARC["test"]
if SMOKE_TEST:
    rs = np.random.default_rng(0)
    pi = np.sort(rs.choice(len(POOL_IMGS), SMOKE_POOL, replace=False)); ti = np.sort(rs.choice(len(TEST_IMGS), SMOKE_TEST_N, replace=False))
    POOL_IMGS, POOL_MASKS, POOL_ITEMS = POOL_IMGS[pi], POOL_MASKS[pi], [POOL_ITEMS[i] for i in pi]
    TEST_IMGS, TEST_MASKS, TEST_ITEMS = TEST_IMGS[ti], TEST_MASKS[ti], [TEST_ITEMS[i] for i in ti]
N_POOL = len(POOL_IMGS)
MIN_AREA_REF = int(np.percentile([g.area for it in POOL_ITEMS for g in it["gts"]], 5))
print(f"pool: {N_POOL} images | test: {len(TEST_IMGS)} images | 5th-percentile lesion area = {MIN_AREA_REF}px")

ARCADE train:   0%|          | 0/1000 [00:00<?, ?it/s]

ARCADE val:   0%|          | 0/200 [00:00<?, ?it/s]

ARCADE test:   0%|          | 0/300 [00:00<?, ?it/s]

train: 1000 images | 1625 lesions | 1.62 per image
val  : 200 images | 406 lesions | 2.03 per image
test : 300 images | 386 lesions | 1.29 per image
pool: 1200 images | test: 300 images | 5th-percentile lesion area = 331px


## Post-processing search (numpy only) on out-of-fold predictions
For each image we keep predicted lesions (connected blobs) that pass: probability **threshold**, **min blob size**, **score cut-off τ**, **relative cut-off** (vs the best blob), **max lesions k**, optional **top1** (force ≥ 1 lesion). The score of a blob is its mean probability. All choices are made on the 1200 out-of-fold images, never on test.

In [7]:
# ============================================================
# Candidate lesions per image + fast evaluation of one post-processing config
# ============================================================
def build_cands(probs, gtu, items, thr, min_area):
    """probs: (N,H,W) float; gtu: (N,H,W) bool GT union. One dict per image with blobs sorted by score."""
    out = []
    for k in range(len(probs)):
        p = probs[k].astype(np.float32)
        n, lab, stats, _ = cv2.connectedComponentsWithStats((p > thr).astype(np.uint8), connectivity=8)
        preds = []
        for i in range(1, n):
            x, y, w, h, a = stats[i]
            if a < min_area: continue
            m = lab[y:y + h, x:x + w] == i
            preds.append(Inst(y, x, m, score=float(p[y:y + h, x:x + w][m].mean())))
        preds.sort(key=lambda t: -t.score)
        gts = items[k]["gts"]
        D = np.zeros((len(preds), len(gts)), np.float32)
        for i, pr in enumerate(preds):
            for j, g in enumerate(gts): D[i, j] = inst_dice(pr, g)
        inter = np.array([np.count_nonzero(pr.m & gtu[k][pr.y0:pr.y1, pr.x0:pr.x1]) for pr in preds], np.int64)
        out.append(dict(preds=preds, scores=np.array([pr.score for pr in preds]), area=np.array([pr.area for pr in preds], np.int64),
                        inter=inter, D=D, gt_area=int(gtu[k].sum()), ngt=len(gts)))
    return out

def keep_idx(sc, tau, rel, max_k, top1):
    n = len(sc)
    keep = [i for i in range(n) if sc[i] >= tau and sc[i] >= rel * sc[0]][:max_k]
    if not keep and top1 and n: keep = [0]
    return keep

def p1_from_D(Dsub, n_gt):
    n_pred = Dsub.shape[0]
    if n_pred == 0 and n_gt == 0: return 1.0
    scores, matched_p, matched_g = [], set(), set()
    if n_pred and n_gt:
        ii, jj = np.nonzero(Dsub > 0)
        for o in np.argsort(-Dsub[ii, jj], kind="stable"):
            i, j = int(ii[o]), int(jj[o])
            if i not in matched_p and j not in matched_g:
                scores.append(float(Dsub[i, j])); matched_p.add(i); matched_g.add(j)
    scores += [0.0] * ((n_pred - len(matched_p)) + (n_gt - len(matched_g)))
    return float(np.mean(scores))

def eval_cfg(C, tau, rel, max_k, top1):
    d_sum = p_sum = 0.0
    for c in C:
        keep = keep_idx(c["scores"], tau, rel, max_k, top1)
        P = int(c["area"][keep].sum()) if keep else 0; I = int(c["inter"][keep].sum()) if keep else 0
        d_sum += 1.0 if P + c["gt_area"] == 0 else 2.0 * I / (P + c["gt_area"])
        p_sum += p1_from_D(c["D"][keep] if keep else np.zeros((0, c["ngt"]), np.float32), c["ngt"])
    return d_sum / len(C), p_sum / len(C)

def select_insts(c, cfg):
    return [c["preds"][i] for i in keep_idx(c["scores"], cfg["tau"], cfg["rel"], cfg["max_k"], cfg["top1"])]

GRID = dict(thr=[0.30, 0.35, 0.40, 0.45, 0.50, 0.55], min_area=sorted({150, MIN_AREA_REF}),
            tau=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8], rel=[0.0, 0.5, 0.75], max_k=[1, 2, 3, 99], top1=[True, False])

def search_postproc(probs, gtu, items, grid=GRID, verbose=True):
    rows = []
    for thr, ma in itertools.product(grid["thr"], grid["min_area"]):
        C = build_cands(probs, gtu, items, thr, ma)
        for tau, rel, mk, t1 in itertools.product(grid["tau"], grid["rel"], grid["max_k"], grid["top1"]):
            d, p = eval_cfg(C, tau, rel, mk, t1)
            rows.append(dict(thr=thr, min_area=ma, tau=tau, rel=rel, max_k=mk, top1=t1, dice_img=d, p1=p))
        if verbose: print(f"  thr={thr:.2f} min_area={ma}: done", end="\r")
    return pd.DataFrame(rows)

def pick_ops(df):
    """Six labelled operating points: objective x (ARCADE protocol with top1 allowed | deployable without top1)."""
    ops = {}
    for tag, sub in [("ARCADE", df), ("DEPLOY", df[~df.top1])]:
        for obj, key in [("maxDice", df.dice_img), ("maxF1", df.p1), ("balanced", (df.dice_img + df.p1) / 2)]:
            s = (key.loc[sub.index]); r = df.loc[s.idxmax()]
            ops[f"{obj}-{tag}"] = dict(thr=float(r.thr), min_area=int(r.min_area), tau=float(r.tau), rel=float(r.rel),
                                       max_k=int(r.max_k), top1=bool(r.top1), oof_dice=float(r.dice_img), oof_p1=float(r.p1))
    return ops
print("post-processing search ready; grid size per (thr,min_area):",
      len(GRID["tau"]) * len(GRID["rel"]) * len(GRID["max_k"]) * len(GRID["top1"]), "| (thr,min_area) sets:", len(GRID["thr"]) * len(GRID["min_area"]))

post-processing search ready; grid size per (thr,min_area): 168 | (thr,min_area) sets: 12


## Vessel masks from the ARCADE *syntax* sub-task (only loaded when the vessel head is switched on)
Same loader and the same safety checks as in the first notebook (non-empty masks, near-duplicates of stenosis test images removed).

In [8]:
# ============================================================
# Load ARCADE-syntax (vessel masks = union of all vessel-segment polygons). Train split is used for the auxiliary task, val split only to monitor it.
# ============================================================
def find_syntax_root():
    cand = DATASET_ROOT.parent / "syntax"
    if (cand / "train" / "annotations" / "train.json").exists(): return cand
    for p in DATASET_ROOT.parent.parent.rglob("train.json"):
        if "syntax" in str(p).lower(): return p.parent.parent.parent
    return None

def load_syntax(root, split, limit=None):
    coco = json.load(open(root / split / "annotations" / f"{split}.json"))
    anns = {}
    for a in coco["annotations"]: anns.setdefault(a["image_id"], []).append(a)
    ims = sorted(coco["images"], key=lambda d: d["id"])
    if limit: ims = ims[:limit]
    imgs, masks = [], []
    for im in tqdm(ims, desc=f"ARCADE-syntax {split}"):
        g = cv2.imread(str(root / split / "images" / im["file_name"]), cv2.IMREAD_GRAYSCALE)
        if g is None: continue
        h, w = g.shape; m = np.zeros((h, w), np.uint8)
        for a in anns.get(im["id"], []): m |= polys_to_mask(a.get("segmentation"), h, w)
        if (h, w) != (IMG_SIZE, IMG_SIZE):
            g = cv2.resize(g, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA); m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        imgs.append(g); masks.append(m)
    return np.stack(imgs), np.stack(masks)

def thumbs(imgs, s=32):
    t = np.stack([cv2.resize(g, (s, s), interpolation=cv2.INTER_AREA) for g in imgs]).reshape(len(imgs), -1).astype(np.float32)
    t -= t.mean(1, keepdims=True); t /= (np.linalg.norm(t, axis=1, keepdims=True) + 1e-6); return t

SYN_IMGS = SYN_MASKS = SYN_VAL_IMGS = SYN_VAL_MASKS = None
if USE_AUX:
    SYNTAX_ROOT = find_syntax_root()
    if SYNTAX_ROOT is None: raise FileNotFoundError("ARCADE 'syntax' folder not found next to 'stenosis'. The new model needs it (or set USE_AUX = False for the refinement-only ablation).")
    print("ARCADE-syntax root:", SYNTAX_ROOT)
    SYN_IMGS, SYN_MASKS = load_syntax(SYNTAX_ROOT, "train", SMOKE_SYN if SMOKE_TEST else None)
    SYN_VAL_IMGS, SYN_VAL_MASKS = load_syntax(SYNTAX_ROOT, "val", 20 if SMOKE_TEST else None)
    nonempty = float((SYN_MASKS.reshape(len(SYN_MASKS), -1).max(1) > 0).mean()); cover = float(SYN_MASKS.mean())
    print(f"syntax train: {len(SYN_IMGS)} images | non-empty vessel masks: {nonempty * 100:.1f}% | vessel pixels: {cover * 100:.2f}% of the image | syntax val (monitor only): {len(SYN_VAL_IMGS)}")
    assert nonempty > 0.9, "Most vessel masks are empty - wrong folder or wrong annotation format. Stop and check."
    # remove syntax images that are (near-)identical to a stenosis TEST image (always checked against the full test set)
    corr = thumbs(SYN_IMGS) @ thumbs(ARC["test"][0]).T; mx = corr.max(1); dup = mx >= 0.98
    print(f"duplicate check vs the {corr.shape[1]} stenosis TEST images: max similarity {mx.max():.3f}, 99th percentile {np.percentile(mx, 99):.3f} -> removed {int(dup.sum())} syntax images (similarity >= 0.98)")
    SYN_IMGS, SYN_MASKS = SYN_IMGS[~dup], SYN_MASKS[~dup]
    fig, ax = plt.subplots(1, 4, figsize=(16, 4.2))
    for a, i in zip(ax, np.linspace(0, len(SYN_IMGS) - 1, 4).astype(int)):
        o = cv2.cvtColor(SYN_IMGS[i], cv2.COLOR_GRAY2RGB); o[SYN_MASKS[i] > 0] = (0.5 * o[SYN_MASKS[i] > 0] + 0.5 * np.array([0, 255, 0])).astype(np.uint8); a.imshow(o); a.axis("off")
    plt.suptitle("SS-N0: auxiliary vessel masks (green) - they must follow the arteries"); plt.tight_layout(); plt.show()
else: print("USE_AUX = False -> auxiliary vessel task disabled (ablation)")

USE_AUX = False -> auxiliary vessel task disabled (ablation)


## Model and training code - **identical to the first notebook**
`VGNet(aux, refine)`: `aux` = auxiliary vessel head, `refine` = vessel-guided refinement head. `aux=False, refine=False` is the plain DeepLabV3+ baseline; `True, True` is the full VG-DeepLab. This notebook trains the configuration chosen by `ABLATION`.

In [9]:
class VGNet(nn.Module):
    """DeepLabV3+ R101. aux: extra vessel channel. refine: vessel-guided full-resolution refinement of the stenosis logit.
    forward(x in [0,1], 3 identical channels) -> (stenosis logit (refined if refine), coarse stenosis logit, vessel logit or None)."""
    def __init__(self, encoder_weights="imagenet", aux=True, refine=True):
        super().__init__()
        self.aux, self.refine = aux, refine
        self.net = smp.DeepLabV3Plus(encoder_name="resnet101", encoder_weights=encoder_weights, in_channels=3, classes=2 if aux else 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        if refine:
            cin = 3 if aux else 2
            self.ref = nn.Sequential(nn.Conv2d(cin, 24, 3, padding=1), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=4, dilation=4), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(24, 1, 1))
            nn.init.zeros_(self.ref[-1].weight); nn.init.zeros_(self.ref[-1].bias)      # starts as identity (no correction)
    def forward(self, x):
        out = self.net((x - self.mean) / self.std)
        s = out[:, :1]; v = out[:, 1:2] if self.aux else None
        if not self.refine: return s, s, v
        f = [x[:, :1] * 2 - 1, torch.sigmoid(s.detach()) * 2 - 1]
        if self.aux: f.append(torch.sigmoid(v.detach()) * 2 - 1)
        return s + self.ref(torch.cat(f, 1)), s, v

def tversky_loss(logits, targets, alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA, eps=1e-6):
    p = torch.sigmoid(logits).view(logits.size(0), -1); t = targets.view(targets.size(0), -1)
    TP = (p * t).sum(1); FP = (p * (1 - t)).sum(1); FN = ((1 - p) * t).sum(1)
    return 1 - ((TP + eps) / (TP + alpha * FP + beta * FN + eps)).mean()

class SegLoss(nn.Module):          # stenosis loss - unchanged from the baseline
    def __init__(self):
        super().__init__(); self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(POS_WEIGHT)))
    def forward(self, lo, t): return 0.5 * self.bce(lo, t) + 0.5 * tversky_loss(lo, t)

def vessel_loss(lo, t):            # auxiliary loss: BCE + soft Dice
    p = torch.sigmoid(lo).view(lo.size(0), -1); tt = t.view(t.size(0), -1)
    dice = 1 - ((2 * (p * tt).sum(1) + 1.0) / (p.sum(1) + tt.sum(1) + 1.0)).mean()
    return 0.5 * F.binary_cross_entropy_with_logits(lo, t) + 0.5 * dice

def augment(g, m):
    """Light geometric + intensity augmentation with plain cv2/numpy (same recipe as the baseline)."""
    h, w = g.shape
    if random.random() < 0.5: g, m = g[:, ::-1], m[:, ::-1]
    g = np.ascontiguousarray(g); m = np.ascontiguousarray(m.astype(np.uint8))
    if random.random() < 0.7:
        M = np.vstack([cv2.getRotationMatrix2D((w / 2, h / 2), random.uniform(-10, 10), random.uniform(0.85, 1.15)), [0, 0, 1]])
        S = np.array([[1, math.tan(math.radians(random.uniform(-2, 2))), 0], [0, 1, 0], [0, 0, 1]], np.float64)
        C = np.array([[1, 0, w / 2], [0, 1, h / 2], [0, 0, 1]], np.float64)
        T = C @ S @ np.linalg.inv(C) @ M
        T[0, 2] += random.uniform(-0.1, 0.1) * w; T[1, 2] += random.uniform(-0.1, 0.1) * h
        g = cv2.warpAffine(g, T[:2], (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        m = cv2.warpAffine(m, T[:2], (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    if random.random() < 0.5:
        g = np.clip(g.astype(np.float32) * (1 + random.uniform(-0.15, 0.15)) + random.uniform(-0.15, 0.15) * 255, 0, 255).astype(np.uint8)
    if random.random() < 0.3:
        g = (np.linspace(0, 1, 256) ** random.uniform(0.8, 1.2) * 255).astype(np.uint8)[g]
    if random.random() < 0.2:
        g = np.clip(g + np.random.default_rng(random.getrandbits(32)).normal(0, random.uniform(3, 12), g.shape), 0, 255).astype(np.uint8)
    return g, m

class ArrDS(Dataset):
    def __init__(self, imgs, masks, train=False): self.imgs, self.masks, self.train = imgs, masks, train
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        g, m = self.imgs[i], self.masks[i]
        if self.train: g, m = augment(g, m)
        img = np.repeat(g[..., None], 3, 2)
        return torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1), torch.from_numpy((m > 0).astype(np.float32)).unsqueeze(0)

class EMA:
    def __init__(self, model, decay):
        self.m = copy.deepcopy(model).eval(); self.decay = decay
        for p in self.m.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step)); msd = model.state_dict()
        for k, v in self.m.state_dict().items():
            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else: v.copy_(msd[k])

def batch_dice(logits, targets, thr=0.5, eps=1e-6):
    p = (torch.sigmoid(logits) > thr).float().view(logits.size(0), -1); t = targets.view(targets.size(0), -1)
    return ((2 * (p * t).sum(1) + eps) / (p.sum(1) + t.sum(1) + eps)).mean().item()

TIMING = {}      # seconds per training step, per model (used for the run-time estimate after the smoke test)
def _forever(dl):
    while True:
        for b in dl: yield b

def train_fold(tag, fold, tr_idx, va_idx, aux, refine):
    seed_everything(TRAIN_SEED + fold)      # new training seed; the folds themselves still use SEED = 42
    tl = DataLoader(ArrDS(POOL_IMGS[tr_idx], POOL_MASKS[tr_idx], train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                    drop_last=True, pin_memory=USE_CUDA, persistent_workers=True)
    vl = DataLoader(ArrDS(POOL_IMGS[va_idx], POOL_MASKS[va_idx]), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    sl = sit = None
    if aux:
        sl = DataLoader(ArrDS(SYN_IMGS, SYN_MASKS, train=True), batch_size=SYN_BATCH, shuffle=True, num_workers=1, drop_last=True, pin_memory=USE_CUDA, persistent_workers=True)
        sit = _forever(sl)
    model = VGNet("imagenet", aux, refine).to(DEVICE); crit = SegLoss().to(DEVICE); ema = EMA(model, EMA_DECAY)
    enc = list(model.net.encoder.parameters()); ids = {id(p) for p in enc}; dec = [p for p in model.parameters() if id(p) not in ids]
    opt = torch.optim.AdamW([{"params": enc, "lr": LR_ENCODER}, {"params": dec, "lr": LR_DECODER}], weight_decay=1e-4)
    total, warm = EPOCHS * len(tl), max(1, WARMUP_EPOCHS * len(tl))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.01, 0.5 * (1 + math.cos(math.pi * min(s, total) / total))))
    scaler = GradScaler("cuda", enabled=USE_CUDA)
    best, best_state, step, t0 = -1.0, None, 0, time.time()
    for ep in range(1, EPOCHS + 1):
        model.train(); tot, totv, nv, t_ep = 0.0, 0.0, 0, time.time()
        for x, y in tl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True); opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=USE_CUDA): s2, s, _ = model(x)
            loss = crit(s2.float(), y)
            if refine: loss = loss + DS_WEIGHT * crit(s.float(), y)
            scaler.scale(loss).backward()
            if aux and step % SYN_EVERY == 0:              # auxiliary vessel batch (gradients are accumulated before the optimiser step)
                xs, ys = next(sit); xs, ys = xs.to(DEVICE, non_blocking=True), ys.to(DEVICE, non_blocking=True)
                with autocast("cuda", enabled=USE_CUDA): _, _, v = model(xs)
                lv = LAMBDA_VESSEL * vessel_loss(v.float(), ys); scaler.scale(lv).backward(); totv += lv.item(); nv += 1
            scaler.step(opt); scaler.update(); sched.step()
            ema.update(model, step); step += 1; tot += loss.item() * x.size(0)
        TIMING[tag] = min(TIMING.get(tag, 1e9), (time.time() - t_ep) / len(tl))
        ds, n = 0.0, 0
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with autocast("cuda", enabled=USE_CUDA): lo = ema.m(x)[0]
                ds += batch_dice(lo.float(), y) * x.size(0); n += x.size(0)
        vd = ds / n
        if vd > best: best = vd; best_state = {k: v.detach().cpu().clone() for k, v in ema.m.state_dict().items()}
        el = time.time() - t0
        print(f"[{tag} f{fold}] ep {ep:2d}/{EPOCHS} loss {tot / (len(tl) * BATCH_SIZE):.4f}" + (f" | vessel loss {totv / max(nv, 1):.4f}" if aux else "") +
              f" | val Dice (EMA) {vd:.4f} | best {best:.4f} | {el:.0f}s (~{el / ep * (EPOCHS - ep):.0f}s left in this fold)" +
              (f" | GPU memory peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB" if (USE_CUDA and ep == 1) else ""))
    del tl, vl, sl, sit, model, ema, opt; torch.cuda.empty_cache()
    return best_state, best

@torch.no_grad()
def predict_probs(model, imgs, bs=8):
    """Test-time augmentation 'ms' (2 scales 1.0 / 1.25 x horizontal flip), the mode chosen on out-of-fold data in both earlier runs.
    Returns stenosis probability (N,512,512) float16 and vessel probability uint8 (or None)."""
    model.eval(); S, V = [], []
    for st in range(0, len(imgs), bs):
        x = torch.from_numpy(imgs[st:st + bs]).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1).to(DEVICE); acc_s, acc_v = 0.0, 0.0
        for sc in (1.0, 1.25):
            xs = x if sc == 1.0 else F.interpolate(x, scale_factor=sc, mode="bilinear", align_corners=False)
            for flip in (False, True):
                xi = torch.flip(xs, dims=[3]) if flip else xs
                with autocast("cuda", enabled=USE_CUDA): s2, _, v = model(xi)
                outs = [torch.sigmoid(s2.float())] + ([torch.sigmoid(v.float())] if v is not None else [])
                if flip: outs = [torch.flip(o, dims=[3]) for o in outs]
                if sc != 1.0: outs = [F.interpolate(o, size=x.shape[-2:], mode="bilinear", align_corners=False) for o in outs]
                acc_s = acc_s + outs[0]
                if v is not None: acc_v = acc_v + outs[1]
        S.append((acc_s / 4)[:, 0].cpu().numpy().astype(np.float16))
        if model.aux: V.append((acc_v / 4)[:, 0].mul(255).round().clamp(0, 255).byte().cpu().numpy())
    return np.concatenate(S), (np.concatenate(V) if V else None)

@torch.no_grad()
def vessel_val_dice(model):
    """Monitor of the auxiliary task on the syntax VAL split (never used for training)."""
    d = []
    for st in range(0, len(SYN_VAL_IMGS), 8):
        x = torch.from_numpy(SYN_VAL_IMGS[st:st + 8]).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1).to(DEVICE)
        with autocast("cuda", enabled=USE_CUDA): v = model(x)[2]
        pr = (torch.sigmoid(v.float())[:, 0] > 0.5).cpu().numpy(); gt = SYN_VAL_MASKS[st:st + 8] > 0
        inter = (pr & gt).sum((1, 2)); den = pr.sum((1, 2)) + gt.sum((1, 2)); d += list(np.where(den == 0, 1.0, 2 * inter / np.maximum(den, 1)))
    return float(np.mean(d))
_n_new, _n_base = (sum(p.numel() for p in VGNet(None, a_, r_).parameters()) for a_, r_ in [(USE_AUX, USE_REFINE), (False, False)])
print(f"model / training code ready | VG-DeepLab parameters: {_n_new / 1e6:.2f}M | baseline: {_n_base / 1e6:.2f}M | extra: {(_n_new - _n_base) / 1e3:.1f}k (the new parts are tiny; the gain, if any, comes from the extra supervision)")

model / training code ready | VG-DeepLab parameters: 45.67M | baseline: 45.67M | extra: 0.0k (the new parts are tiny; the gain, if any, comes from the extra supervision)


## Cross-validation training of the ablation model (same 5 folds, same seed as the first run)
Checkpoints are saved after each fold. Models that already exist in the attached inputs are loaded instead of trained.

In [10]:
y_strat = np.array([min(len(it["gts"]), 3) for it in POOL_ITEMS])
FOLDS = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(N_POOL), y_strat))
GTU_POOL = POOL_MASKS > 0
NEW_TAG = ("vg_dl3" if WHICH == "N" else "dl3_r101") + f"_seed{SEED_RUN}"

# checkpoints that already exist: this notebook's own run folder first, then anything attached under /kaggle/input (smoke-test checkpoints are ignored)
INPUT_CKPTS = {}
for _root in [Path("/kaggle/input"), Path(os.environ.get("VG_CKPT_DIR", "/nonexistent"))]:
    if _root.exists():
        for _p in sorted(_root.rglob("*_f[0-9].pt")):
            if "smoke" not in str(_p).lower(): INPUT_CKPTS.setdefault(_p.name, _p)
print("checkpoints found in the attached inputs:", sorted(INPUT_CKPTS) if INPUT_CKPTS else "NONE (did you add the output of the first VG-DeepLab notebook as an input?)")
def ckpt_path(tag, fold):
    p = RUN_DIR / f"{tag}_f{fold}.pt"
    return p if p.exists() else INPUT_CKPTS.get(p.name)
_need = [f"{t}_f{f}.pt" for t in ["dl3_r101", "vg_dl3"] for f in range(N_FOLDS) if ckpt_path(t, f) is None]
if _need and REQUIRE_FIRST_RUN:
    raise FileNotFoundError(f"Checkpoints of the first run not found: {_need}. On Kaggle: Add Input -> Your Work / Notebooks -> 'ARCADE Stenosis — VG-DeepLab' (output of the first run), then run again. "
                            "(Only if that output no longer exists: set REQUIRE_FIRST_RUN = False in the config cell; the baseline is then trained here and the full model is left out.)")

def run_cv(tag, aux, refine):
    oof = np.zeros((N_POOL, IMG_SIZE, IMG_SIZE), np.float16); log, t_all = [], time.time()
    for fold, (tr, va) in enumerate(FOLDS):
        ck = ckpt_path(tag, fold)
        if ck is not None:
            print(f"[{tag} f{fold}] checkpoint found ({ck}), not training"); state = torch.load(ck, map_location="cpu"); bd = float("nan")
        else:
            state, bd = train_fold(tag, fold, tr, va, aux, refine); torch.save(state, RUN_DIR / f"{tag}_f{fold}.pt")
        net = VGNet(None, aux, refine).to(DEVICE); net.load_state_dict(state); net.eval()
        oof[va] = predict_probs(net, POOL_IMGS[va])[0]
        row = dict(model=tag, fold=fold, n_train=len(tr), n_val=len(va), best_val_dice=bd)
        if aux and SYN_VAL_IMGS is not None: row["vessel_dice_syntax_val"] = vessel_val_dice(net)
        log.append(row); del net, state; torch.cuda.empty_cache()
        print(f"=== finished {tag} fold {fold + 1}/{N_FOLDS} | total elapsed {(time.time() - t_all) / 60:.1f} min ===")
    return oof, pd.DataFrame(log)

OOF_NEW, LOG_NEW = run_cv(NEW_TAG, USE_AUX, USE_REFINE)
print(f"=== SS-A1: ablation model {NEW_TAG}, per-fold log ===")
print(LOG_NEW.round(4).to_string(index=False))
def eta_hours(tag):
    n_full = len(ARC["train"][0]) + len(ARC["val"][0]); folds, ep = PRESETS[MODE]["N_FOLDS"], PRESETS[MODE]["EPOCHS"]
    return TIMING[tag] * (n_full * (folds - 1) / folds / BATCH_SIZE) * ep * folds / 3600
if NEW_TAG in TIMING: print(f"measured {TIMING[NEW_TAG]:.2f} s per training step -> a full run (SMOKE_TEST = False, MODE = '{MODE}') of this ablation model needs about {eta_hours(NEW_TAG):.1f} h of training on this GPU (rough; first-epoch timing is pessimistic)")

checkpoints found in the attached inputs: ['dl3_r101_f0.pt', 'dl3_r101_f1.pt', 'dl3_r101_f2.pt', 'dl3_r101_f3.pt', 'dl3_r101_f4.pt', 'vg_dl3_f0.pt', 'vg_dl3_f1.pt', 'vg_dl3_f2.pt', 'vg_dl3_f3.pt', 'vg_dl3_f4.pt']


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  179MB            

model.safetensors: downloading bytes:           |  0.00B            

[dl3_r101_seed1 f0] ep  1/40 loss 0.6450 | val Dice (EMA) 0.2727 | best 0.2727 | 58s (~2259s left in this fold) | GPU memory peak 4.8 GB
[dl3_r101_seed1 f0] ep  2/40 loss 0.4384 | val Dice (EMA) 0.3820 | best 0.3820 | 102s (~1938s left in this fold)
[dl3_r101_seed1 f0] ep  3/40 loss 0.3527 | val Dice (EMA) 0.4687 | best 0.4687 | 146s (~1799s left in this fold)
[dl3_r101_seed1 f0] ep  4/40 loss 0.3132 | val Dice (EMA) 0.4996 | best 0.4996 | 190s (~1712s left in this fold)
[dl3_r101_seed1 f0] ep  5/40 loss 0.2969 | val Dice (EMA) 0.5294 | best 0.5294 | 235s (~1645s left in this fold)
[dl3_r101_seed1 f0] ep  6/40 loss 0.2863 | val Dice (EMA) 0.5223 | best 0.5294 | 280s (~1586s left in this fold)
[dl3_r101_seed1 f0] ep  7/40 loss 0.2651 | val Dice (EMA) 0.5394 | best 0.5394 | 325s (~1533s left in this fold)
[dl3_r101_seed1 f0] ep  8/40 loss 0.2562 | val Dice (EMA) 0.5465 | best 0.5465 | 370s (~1482s left in this fold)
[dl3_r101_seed1 f0] ep  9/40 loss 0.2477 | val Dice (EMA) 0.5619 | best 

## Reference models (already trained in your first VG-DeepLab run)
The baseline (`dl3_r101_f*.pt`) and the full VG-DeepLab (`vg_dl3_f*.pt`) are **loaded from the attached output of your first notebook** - they are not trained again. The notebook re-computes their out-of-fold and test predictions with exactly the same code and checks that the numbers match the first run (a mismatch would mean wrong files).

In [11]:
try:
    # label, checkpoint tag, aux, refine   (first letter of the label is used as a short name: B / V / R / N)
    ALL_METHODS = [("B baseline (DeepLabV3+)", "dl3_r101", False, False), (f"S {WHICH} with training seed {TRAIN_SEED}", NEW_TAG, USE_AUX, USE_REFINE), ("N full VG-DeepLab", "vg_dl3", True, True)]
    OOFS, SPEC, LOGS = {}, {}, {}
    for label, tag, aux, refine in ALL_METHODS:
        if tag == NEW_TAG:
            OOFS[label], LOGS[label] = OOF_NEW, LOG_NEW; SPEC[label] = (tag, aux, refine); continue
        have = all(ckpt_path(tag, f) is not None for f in range(N_FOLDS))
        if have or (tag == "dl3_r101" and TRAIN_BASELINE_IF_MISSING):
            if not have: print(f"!! {tag}: checkpoints not found in the attached inputs -> training it here (+~2.6 h). Attach the output of your first VG-DeepLab notebook to avoid this.")
            OOFS[label], LOGS[label] = run_cv(tag, aux, refine); SPEC[label] = (tag, aux, refine)
        else: print(f"{label}: no checkpoints found ({tag}_f*.pt) -> not included in the comparison")
    KEYS = [l for l, *_ in ALL_METHODS if l in OOFS]
    KB = "B baseline (DeepLabV3+)" if "B baseline (DeepLabV3+)" in OOFS else None
    KA = next(l for l, t, *_ in ALL_METHODS if t == NEW_TAG)           # the model trained in this notebook
    KFULL = "N full VG-DeepLab" if "N full VG-DeepLab" in OOFS else None
    PAIRS = ([(KB, k) for k in KEYS if k != KB] if KB else []) + ([(k, KFULL) for k in KEYS if k not in (KB, KFULL)] if KFULL else [])
    print("\nmethods in this comparison:", [k.split(" (")[0] for k in KEYS])
    print("paired tests that will be run:", [f"{b[0]} minus {a[0]}" for a, b in PAIRS])
    if KB is None: print("WARNING: no baseline -> the ablation cannot be compared with the plain DeepLabV3+.")
except Exception:
    import traceback; traceback.print_exc(); ANALYSIS_ERRORS.append('analysis cell 1'); print('!! ERROR in analysis cell 1 (training results are safe; see the last cell)')

[dl3_r101 f0] checkpoint found (/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f0.pt), not training
=== finished dl3_r101 fold 1/5 | total elapsed 0.4 min ===
[dl3_r101 f1] checkpoint found (/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f1.pt), not training
=== finished dl3_r101 fold 2/5 | total elapsed 0.8 min ===
[dl3_r101 f2] checkpoint found (/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f2.pt), not training
=== finished dl3_r101 fold 3/5 | total elapsed 1.1 min ===
[dl3_r101 f3] checkpoint found (/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f3.pt), not training
=== finished dl3_r101 fold 4/5 | total elapsed 1.6 min ===
[dl3_r101 f4] checkpoint found (/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f4.pt), not training
=== finished dl3_r101 fold 5/5 | total elapsed 1.9 min ===
[vg_dl3 f0] checkpoint found (

## Out-of-fold comparison (1200 images, no test data)
Same procedure as the first notebook: the post-processing is searched separately for each method on its own out-of-fold predictions, then all methods are compared at the `balanced-DEPLOY` operating point.

In [12]:
try:
    CAND_THR, CAND_MIN_AREA, CAND_MAX = 0.30, 100, 8
    N_BOOT_MAP = 20 if SMOKE_TEST else 200
    HEADLINE = "balanced-DEPLOY"
    EXPECT = {"B": dict(oof_dice=0.5941, test_dice=0.6197, test_f1=0.5165), "N": dict(oof_dice=0.6028, test_dice=0.6358, test_f1=0.5436)}   # numbers of the first run

    def pixel_confusion(preds_by_img, items):
        TP = FP = FN = total = 0
        for preds, it in zip(preds_by_img, items):
            h, w = it["shape"]; total += h * w
            pr, gt = paint(preds, it["shape"]), paint(it["gts"], it["shape"])
            TP += int(np.count_nonzero(pr & gt)); FP += int(np.count_nonzero(pr & ~gt)); FN += int(np.count_nonzero(~pr & gt))
        TN = total - TP - FP - FN
        return dict(TP=TP, FP=FP, FN=FN, TN=TN, precision=TP / max(TP + FP, 1), recall=TP / max(TP + FN, 1), specificity=TN / max(TN + FP, 1),
                    dice=2 * TP / max(2 * TP + FP + FN, 1), iou=TP / max(TP + FP + FN, 1))

    def inst_counts(preds_by_img, items):
        tp, npred, ngt = [], [], []
        for preds, it in zip(preds_by_img, items):
            pairs, _, _ = inst_greedy_match(preds, it["gts"], "iou")
            tp.append(sum(1 for _, _, s in pairs if s >= 0.5)); npred.append(len(preds)); ngt.append(len(it["gts"]))
        return np.array(tp), np.array(npred), np.array(ngt)

    def f1_from_counts(cnt, idx=None):
        tp, npred, ngt = cnt
        if idx is not None: tp, npred, ngt = tp[idx], npred[idx], ngt[idx]
        return 2 * tp.sum() / max(npred.sum() + ngt.sum(), 1)

    def make_cands(probs, gtu, items, thr=CAND_THR, min_area=CAND_MIN_AREA, max_per_img=CAND_MAX):
        C = build_cands(probs, gtu, items, thr, min_area); out = []
        for c in C:
            n = min(len(c["preds"]), max_per_img)
            out.append(dict(preds=c["preds"][:n], scores=c["scores"][:n], area=c["area"][:n], inter=c["inter"][:n], D=c["D"][:n], gt_area=c["gt_area"], ngt=c["ngt"]))
        return out

    def map_prep(C, items):
        ious = []
        for c, it in zip(C, items):
            m = np.zeros((len(c["preds"]), len(it["gts"])), np.float32)
            for i, p in enumerate(c["preds"]):
                for j, g in enumerate(it["gts"]): m[i, j] = inst_iou(p, g)
            ious.append(m)
        return ious, np.array([len(it["gts"]) for it in items]), [np.asarray(c["scores"], np.float64) for c in C]

    def map_from_prep(prep, idx=None, thrs=np.arange(0.50, 0.96, 0.05)):
        ious, ngt, scores = prep
        if idx is None: idx = np.arange(len(ngt))
        n_gt = int(ngt[idx].sum())
        order = sorted(((float(scores[k][i]), j, i) for j, k in enumerate(idx) for i in range(len(scores[k]))), key=lambda d: -d[0])
        aps = []
        for thr in thrs:
            used = [np.zeros(ngt[k], bool) for k in idx]; tp = np.zeros(len(order))
            for n, (_, j, i) in enumerate(order):
                k = idx[j]
                if ngt[k] == 0: continue
                cand = np.where(~used[j], ious[k][i], -1.0); g = int(cand.argmax())
                if cand[g] >= thr: used[j][g] = True; tp[n] = 1
            ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp); rec = ctp / max(n_gt, 1)
            prec = np.maximum.accumulate((ctp / np.maximum(ctp + cfp, 1e-9))[::-1])[::-1] if len(order) else np.zeros(0)
            aps.append(float(np.mean([prec[rec >= r].max() if np.any(rec >= r) else 0.0 for r in np.linspace(0, 1, 101)])))
        return aps[0], float(np.mean(aps))

    def lesion_recall(C):
        hit, best = [], []
        for c in C:
            if c["ngt"] == 0: continue
            Dm = c["D"].max(0) if len(c["preds"]) else np.zeros(c["ngt"]); iou = Dm / (2.0 - Dm); hit += list(iou >= 0.5); best += list(iou)
        return float(np.mean(hit)) if hit else 0.0, float(np.mean(best)) if best else 0.0

    def paired_boot(fn_a, fn_b, n_img, n_boot, seed=0):
        rng = np.random.default_rng(seed); d = []
        for _ in range(n_boot):
            idx = rng.integers(0, n_img, n_img); d.append(fn_b(idx) - fn_a(idx))
        d = np.array(d); return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float((d > 0).mean())

    def ci_mean(v, n=1000, seed=1):
        r = np.random.default_rng(seed); b = [v[r.integers(0, len(v), len(v))].mean() for _ in range(n)]; return np.percentile(b, [2.5, 97.5])

    def apply_op(probs, gtu, items, op):
        C = build_cands(probs, gtu, items, op["thr"], op["min_area"]); return [select_insts(c, op) for c in C]

    def short(k): return k.split(" (")[0]

    t0 = time.time(); OPS_ALL, OOF_EVAL, rows = {}, {}, []
    for k in KEYS:
        oof = OOFS[k]
        df = search_postproc(oof, GTU_POOL, POOL_ITEMS, verbose=False); OPS_ALL[k] = pick_ops(df)
        C = make_cands(oof, GTU_POOL, POOL_ITEMS); rec, miou = lesion_recall(C); ap50, ap5095 = map_from_prep(map_prep(C, POOL_ITEMS))
        preds = apply_op(oof, GTU_POOL, POOL_ITEMS, OPS_ALL[k][HEADLINE]); fm = full_metrics(preds, POOL_ITEMS); OOF_EVAL[k] = fm
        rows.append({"method": short(k), "Dice/img": round(fm["Dice_img"], 4), "official F1": round(fm["P1"], 4), "inst F1@.5": round(fm["F1"], 4), "mAP50": round(ap50, 4), "mAP50-95": round(ap5095, 4),
                     "lesion recall@IoU.5": round(rec, 3), "mean best IoU": round(miou, 3)})
        print(f"  {short(k)}: post-processing searched ({time.time() - t0:.0f}s) -> {OPS_ALL[k][HEADLINE]}")
    TAB_OOF = pd.DataFrame(rows)
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
    print(f"\n=== SS-A2: OUT-OF-FOLD comparison at each method's own '{HEADLINE}' operating point ({N_POOL} images, not test) ===")
    print(TAB_OOF.iloc[:, :6].to_string(index=False)); print(); print(TAB_OOF.iloc[:, [0, 6, 7]].to_string(index=False))
    rows = []
    for ka, kb in PAIRS:
        for met, key in [("Dice/img", "per_image_Dice"), ("official F1", "per_image_P1")]:
            a, b = OOF_EVAL[ka][key], OOF_EVAL[kb][key]; lo, hi, pg = paired_boot(lambda i: a[i].mean(), lambda i: b[i].mean(), N_POOL, 1000)
            rows.append({"comparison": f"{kb[0]} minus {ka[0]}", "metric": met, "difference": round(float(b.mean() - a.mean()), 4), "95% CI": f"[{lo:+.4f}, {hi:+.4f}]", "P(better)": round(pg, 3),
                         "CI excludes 0": "YES" if (lo > 0 or hi < 0) else "no"})
    TAB_OOF_PAIR = pd.DataFrame(rows)
    print(f"\n=== SS-A3: paired bootstrap on the OUT-OF-FOLD images ==="); print(TAB_OOF_PAIR.to_string(index=False) if len(TAB_OOF_PAIR) else "(no pairs)")
    if not SMOKE_TEST:
        for k in KEYS:
            e = EXPECT.get(k[0])
            if e: print(f"reproduction check {k[0]}: out-of-fold Dice/img {OOF_EVAL[k]['Dice_img']:.4f} (first run {e['oof_dice']:.4f})", "OK" if abs(OOF_EVAL[k]["Dice_img"] - e["oof_dice"]) < 0.003 else "<-- MISMATCH: tell Claude")
    json.dump({"headline": HEADLINE, "ops": {k: v for k, v in OPS_ALL.items()}, "new_tag": NEW_TAG, "use_aux": USE_AUX, "use_refine": USE_REFINE, "epochs": EPOCHS, "folds": N_FOLDS},
              open(OUT_DIR / "frozen_choices_ablation.json", "w"), indent=2)
    print("\nFROZEN. Everything above used only train+val. The test set comes next.")
except Exception:
    import traceback; traceback.print_exc(); ANALYSIS_ERRORS.append('analysis cell 2'); print('!! ERROR in analysis cell 2 (training results are safe; see the last cell)')

  B baseline: post-processing searched (147s) -> {'thr': 0.4, 'min_area': 150, 'tau': 0.5, 'rel': 0.75, 'max_k': 99, 'top1': False, 'oof_dice': 0.5940754985405674, 'oof_p1': 0.4653226437734013}
  S B with training seed 1042: post-processing searched (292s) -> {'thr': 0.3, 'min_area': 150, 'tau': 0.4, 'rel': 0.75, 'max_k': 3, 'top1': False, 'oof_dice': 0.594791406254179, 'oof_p1': 0.46215870664538516}
  N full VG-DeepLab: post-processing searched (439s) -> {'thr': 0.35, 'min_area': 150, 'tau': 0.2, 'rel': 0.75, 'max_k': 3, 'top1': False, 'oof_dice': 0.6027725797465645, 'oof_p1': 0.4630665738913597}

=== SS-A2: OUT-OF-FOLD comparison at each method's own 'balanced-DEPLOY' operating point (1200 images, not test) ===
                     method  Dice/img  official F1  inst F1@.5  mAP50  mAP50-95
                 B baseline    0.5941       0.4653      0.4096 0.2349    0.0665
S B with training seed 1042    0.5948       0.4622      0.4070 0.2458    0.0728
          N full VG-DeepLab    0.6028

# TEST SET
Same evaluation as the first notebook, for every method. The test set was already used for the earlier models, so this ablation is **explanatory**: it tells which part of VG-DeepLab is responsible for the gain. Trust only differences whose confidence interval excludes 0.

In [13]:
try:
    GTU_TEST = TEST_MASKS > 0
    def test_ensemble(tag, aux, refine):
        acc = np.zeros(TEST_IMGS.shape, np.float32)
        for fold in range(N_FOLDS):
            net = VGNet(None, aux, refine).to(DEVICE); net.load_state_dict(torch.load(ckpt_path(tag, fold), map_location="cpu")); net.eval()
            acc += predict_probs(net, TEST_IMGS)[0].astype(np.float32); del net; torch.cuda.empty_cache()
        print(f"{tag}: test inference done ({N_FOLDS} fold models averaged)")
        return (acc / N_FOLDS).astype(np.float16)

    ENS = {k: test_ensemble(*SPEC[k]) for k in KEYS}
    RES = {}
    for k, probs in ENS.items():
        r = {"ops": {}}
        for oname in [HEADLINE, "maxF1-ARCADE"]:
            preds = apply_op(probs, GTU_TEST, TEST_ITEMS, OPS_ALL[k][oname]); r["ops"][oname] = dict(preds=preds, fm=full_metrics(preds, TEST_ITEMS))
        h = r["ops"][HEADLINE]; r["pc"] = pixel_confusion(h["preds"], TEST_ITEMS); r["cnt"] = inst_counts(h["preds"], TEST_ITEMS)
        r["prep"] = map_prep(make_cands(probs, GTU_TEST, TEST_ITEMS), TEST_ITEMS); r["map50"], r["map5095"] = map_from_prep(r["prep"]); RES[k] = r
    n_img = len(TEST_ITEMS); rng = np.random.default_rng(0); BOOT_IDX = [rng.integers(0, n_img, n_img) for _ in range(N_BOOT_MAP)]
    for k, r in RES.items():
        bs = [map_from_prep(r["prep"], idx) for idx in BOOT_IDX]
        r["map50_ci"], r["map5095_ci"] = np.percentile([b[0] for b in bs], [2.5, 97.5]), np.percentile([b[1] for b in bs], [2.5, 97.5])
        r["f1_ci"] = np.percentile([f1_from_counts(r["cnt"], idx) for idx in BOOT_IDX], [2.5, 97.5])

    rowsA, rowsB = [], []
    for k, r in RES.items():
        for oname, o in r["ops"].items():
            fm = o["fm"]; dci, pci = ci_mean(fm["per_image_Dice"]), ci_mean(fm["per_image_P1"])
            rowsA.append({"method": short(k), "operating point": oname, "Dice/img [95% CI]": f"{fm['Dice_img']:.4f} [{dci[0]:.3f},{dci[1]:.3f}]", "Dice pooled": round(fm["Dice"], 4),
                          "official F1 [95% CI]": f"{fm['P1']:.4f} [{pci[0]:.3f},{pci[1]:.3f}]"})
            rowsB.append({"method": short(k), "operating point": oname, "inst F1@.5": round(fm["F1"], 4), "P@.5": round(fm["Precision"], 4), "R@.5": round(fm["Recall"], 4),
                          "IoU/img": round(fm["IoU_img"], 4), "lesions/img": round(fm["pred_per_img"], 2)})
    TAB_A, TAB_B = pd.DataFrame(rowsA), pd.DataFrame(rowsB)
    TAB_C = pd.DataFrame([{"method": short(k), "mAP50 [95% CI]": f"{r['map50']:.4f} [{r['map50_ci'][0]:.3f},{r['map50_ci'][1]:.3f}]", "mAP50-95 [95% CI]": f"{r['map5095']:.4f} [{r['map5095_ci'][0]:.3f},{r['map5095_ci'][1]:.3f}]",
                           "inst F1@.5 [95% CI]": f"{f1_from_counts(r['cnt']):.4f} [{r['f1_ci'][0]:.3f},{r['f1_ci'][1]:.3f}]"} for k, r in RES.items()])
    TAB_PC = pd.DataFrame([{"method": short(k), "TP": r["pc"]["TP"], "FP": r["pc"]["FP"], "FN": r["pc"]["FN"], "TN": r["pc"]["TN"], "precision": round(r["pc"]["precision"], 4),
                            "recall": round(r["pc"]["recall"], 4), "specificity": round(r["pc"]["specificity"], 5), "Dice pooled": round(r["pc"]["dice"], 4), "IoU pooled": round(r["pc"]["iou"], 4)} for k, r in RES.items()])
    print(f"=== SS-A4a: TEST - Dice and official ARCADE F1 (headline = {HEADLINE}; the '-ARCADE' row may force one lesion per image) ==="); print(TAB_A.to_string(index=False))
    print("\n=== SS-A4b: TEST - instance-level metrics ==="); print(TAB_B.to_string(index=False))
    print(f"\n=== SS-A4c: TEST - mask mAP and instance F1 at '{HEADLINE}' ==="); print(TAB_C.to_string(index=False))
    print(f"\n=== SS-A4d: TEST - pixel-level confusion at '{HEADLINE}' ==="); print(TAB_PC.to_string(index=False))
    if not SMOKE_TEST:
        for k in KEYS:
            e = EXPECT.get(k[0]); fm = RES[k]["ops"][HEADLINE]["fm"]
            if e: print(f"reproduction check {k[0]}: test Dice/img {fm['Dice_img']:.4f} (first run {e['test_dice']:.4f}), official F1 {fm['P1']:.4f} (first run {e['test_f1']:.4f})",
                        "OK" if (abs(fm["Dice_img"] - e["test_dice"]) < 0.003 and abs(fm["P1"] - e["test_f1"]) < 0.004) else "<-- MISMATCH: tell Claude")
except Exception:
    import traceback; traceback.print_exc(); ANALYSIS_ERRORS.append('analysis cell 3'); print('!! ERROR in analysis cell 3 (training results are safe; see the last cell)')

dl3_r101: test inference done (5 fold models averaged)
dl3_r101_seed1: test inference done (5 fold models averaged)
vg_dl3: test inference done (5 fold models averaged)
=== SS-A4a: TEST - Dice and official ARCADE F1 (headline = balanced-DEPLOY; the '-ARCADE' row may force one lesion per image) ===
                     method operating point    Dice/img [95% CI]  Dice pooled official F1 [95% CI]
                 B baseline balanced-DEPLOY 0.6197 [0.596,0.640]       0.6327 0.5165 [0.489,0.544]
                 B baseline    maxF1-ARCADE 0.6211 [0.597,0.642]       0.6364 0.5230 [0.495,0.550]
S B with training seed 1042 balanced-DEPLOY 0.6223 [0.596,0.644]       0.6359 0.5334 [0.504,0.561]
S B with training seed 1042    maxF1-ARCADE 0.6226 [0.596,0.644]       0.6370 0.5347 [0.505,0.561]
          N full VG-DeepLab balanced-DEPLOY 0.6358 [0.611,0.657]       0.6468 0.5436 [0.515,0.570]
          N full VG-DeepLab    maxF1-ARCADE 0.6412 [0.616,0.662]       0.6521 0.5579 [0.530,0.584]

=== SS-

In [14]:
try:
    # ---- paired bootstrap on the TEST images + the ablation table ----
    rows = []
    for ka, kb in PAIRS:
        A_, B_ = RES[ka], RES[kb]; fa, fb = A_["ops"][HEADLINE]["fm"], B_["ops"][HEADLINE]["fm"]
        tests = [("Dice/img", lambda i: fa["per_image_Dice"][i].mean(), lambda i: fb["per_image_Dice"][i].mean(), 1000),
                 ("official F1", lambda i: fa["per_image_P1"][i].mean(), lambda i: fb["per_image_P1"][i].mean(), 1000),
                 ("inst F1@.5", lambda i: f1_from_counts(A_["cnt"], i), lambda i: f1_from_counts(B_["cnt"], i), 1000),
                 ("mAP50", lambda i: map_from_prep(A_["prep"], i)[0], lambda i: map_from_prep(B_["prep"], i)[0], N_BOOT_MAP),
                 ("mAP50-95", lambda i: map_from_prep(A_["prep"], i)[1], lambda i: map_from_prep(B_["prep"], i)[1], N_BOOT_MAP)]
        for met, f_a, f_b, nb in tests:
            full = np.arange(n_img); diff = f_b(full) - f_a(full); lo, hi, pg = paired_boot(f_a, f_b, n_img, nb)
            rows.append({"comparison": f"{kb[0]} minus {ka[0]}", "metric": met, "difference": round(float(diff), 4), "95% CI": f"[{lo:+.4f}, {hi:+.4f}]",
                         "P(better)": round(pg, 3), "CI excludes 0": "YES" if (lo > 0 or hi < 0) else "no"})
    TAB_PAIR = pd.DataFrame(rows)
    print(f"=== SS-A5: paired bootstrap on the TEST images at '{HEADLINE}' (B = baseline, V = vessel head only, R = refinement only, N = full model) ===")
    print(TAB_PAIR.to_string(index=False) if len(TAB_PAIR) else "(no pairs)")

    g = []
    for k in KEYS:
        tag, aux, refine = SPEC[k]; fm = RES[k]["ops"][HEADLINE]["fm"]; fo = OOF_EVAL[k]
        g.append({"method": short(k), "vessel head": "yes" if aux else "-", "refinement": "yes" if refine else "-", "OOF Dice/img": round(fo["Dice_img"], 4), "OOF off.F1": round(fo["P1"], 4),
                  "TEST Dice/img": round(fm["Dice_img"], 4), "TEST Dice pooled": round(fm["Dice"], 4), "TEST off.F1": round(fm["P1"], 4), "TEST inst F1@.5": round(fm["F1"], 4),
                  "TEST mAP50": round(RES[k]["map50"], 4), "TEST mAP50-95": round(RES[k]["map5095"], 4)})
    GATE = pd.DataFrame(g)
    print("\n=== SS-A6: ABLATION TABLE (this is the table for the thesis) ==="); print(GATE.iloc[:, :5].to_string(index=False)); print(); print(GATE.iloc[:, [0, 5, 6, 7, 8, 9, 10]].to_string(index=False))
    print("\n--- what the data supports ---")
    def verdict(ka, kb, what):
        for src, tab in [("out-of-fold", TAB_OOF_PAIR), ("test", TAB_PAIR)]:
            t = tab[tab.comparison == f"{kb[0]} minus {ka[0]}"] if len(tab) else tab
            w = [f"{r_.metric} {r_.difference:+.4f}" for _, r_ in t.iterrows() if r_["CI excludes 0"] == "YES" and r_.difference > 0]
            l = [f"{r_.metric} {r_.difference:+.4f}" for _, r_ in t.iterrows() if r_["CI excludes 0"] == "YES" and r_.difference < 0]
            print(f"  {what} [{src}]: reliable gain on: {', '.join(w) if w else 'nothing'}" + (f" | reliably WORSE on: {', '.join(l)}" if l else ""))
    if KB:
        verdict(KB, KA, f"{KA[0]} ({short(KA)}) vs baseline")
        if KFULL: verdict(KB, KFULL, "N (full model) vs baseline"); verdict(KA, KFULL, f"N (full model) vs {KA[0]} = what the OTHER component adds")
    print("Rule: say that a component 'helps' only if the out-of-fold AND the test comparison both show a reliable gain. Otherwise write 'no reliable effect'.")
except Exception:
    import traceback; traceback.print_exc(); ANALYSIS_ERRORS.append('analysis cell 4'); print('!! ERROR in analysis cell 4 (training results are safe; see the last cell)')

=== SS-A5: paired bootstrap on the TEST images at 'balanced-DEPLOY' (B = baseline, V = vessel head only, R = refinement only, N = full model) ===
comparison      metric  difference             95% CI  P(better) CI excludes 0
 S minus B    Dice/img      0.0026 [-0.0067, +0.0109]      0.704            no
 S minus B official F1      0.0170 [+0.0035, +0.0303]      0.993           YES
 S minus B  inst F1@.5     -0.0090 [-0.0404, +0.0222]      0.281            no
 S minus B       mAP50      0.0020 [-0.0223, +0.0331]      0.615            no
 S minus B    mAP50-95      0.0015 [-0.0059, +0.0117]      0.690            no
 N minus B    Dice/img      0.0161 [+0.0079, +0.0249]      1.000           YES
 N minus B official F1      0.0271 [+0.0133, +0.0422]      1.000           YES
 N minus B  inst F1@.5      0.0273 [-0.0025, +0.0571]      0.962            no
 N minus B       mAP50      0.0431 [+0.0068, +0.0797]      1.000           YES
 N minus B    mAP50-95      0.0110 [+0.0032, +0.0215]      1.000

In [15]:
try:
    # ---- save everything ----
    TAB_OOF.to_csv(OUT_DIR / "oof_comparison.csv", index=False); TAB_OOF_PAIR.to_csv(OUT_DIR / "oof_paired_bootstrap.csv", index=False)
    TAB_A.to_csv(OUT_DIR / "test_dice_f1.csv", index=False); TAB_B.to_csv(OUT_DIR / "test_instance.csv", index=False); TAB_C.to_csv(OUT_DIR / "test_map.csv", index=False)
    TAB_PC.to_csv(OUT_DIR / "pixel_confusion.csv", index=False); TAB_PAIR.to_csv(OUT_DIR / "paired_bootstrap.csv", index=False); GATE.to_csv(OUT_DIR / "ablation_table.csv", index=False)
    LOG_NEW.to_csv(OUT_DIR / "fold_log_new.csv", index=False)
    per_img = pd.DataFrame({"file": [it["file"] for it in TEST_ITEMS], "n_gt": [len(it["gts"]) for it in TEST_ITEMS]})
    per_oof = pd.DataFrame({"file": [it["file"] for it in POOL_ITEMS], "n_gt": [len(it["gts"]) for it in POOL_ITEMS]})
    for k in KEYS:
        per_img[f"dice_{k[0]}"] = RES[k]["ops"][HEADLINE]["fm"]["per_image_Dice"]; per_img[f"officialF1_{k[0]}"] = RES[k]["ops"][HEADLINE]["fm"]["per_image_P1"]
        per_oof[f"dice_{k[0]}"] = OOF_EVAL[k]["per_image_Dice"]; per_oof[f"officialF1_{k[0]}"] = OOF_EVAL[k]["per_image_P1"]
        tp_, np_, ng_ = RES[k]["cnt"]; per_img[f"tp50_{k[0]}"] = tp_; per_img[f"npred_{k[0]}"] = np_
    per_img.to_csv(OUT_DIR / "per_image_test_metrics.csv", index=False); per_oof.to_csv(OUT_DIR / "per_image_oof_metrics.csv", index=False)
    sfx = "_smoke" if SMOKE_TEST else ""
    np.savez_compressed(BASE / f"{NEW_TAG}_probs{sfx}.npz", test=np.round(ENS[KA].astype(np.float32) * 255).astype(np.uint8), oof=np.round(OOF_NEW.astype(np.float32) * 255).astype(np.uint8))
    ZIP_BASE = str(BASE / f"results_seed_{NEW_TAG}{sfx}"); shutil.make_archive(ZIP_BASE, "zip", OUT_DIR)
    print("Saved tables to", OUT_DIR, "| zip:", ZIP_BASE + ".zip")
except Exception:
    import traceback; traceback.print_exc(); ANALYSIS_ERRORS.append('analysis cell 5'); print('!! ERROR in analysis cell 5 (training results are safe; see the last cell)')

Saved tables to /kaggle/working/outputs | zip: /kaggle/working/results_seed_dl3_r101_seed1.zip


In [16]:
# ---- final status ----
if ANALYSIS_ERRORS:
    print("!" * 100); print("TRAINING FINISHED AND THE CHECKPOINTS ARE SAVED, but these analysis cells had an error:", ANALYSIS_ERRORS)
    print("Do NOT re-train. Send Claude the error text printed above; the analysis can be re-run from the saved checkpoints in a few minutes."); print("!" * 100)
else:
    print("ALL DONE - no errors. Send Claude: the zip 'results_seed_*.zip' from the Output panel (and SS-A2 ... SS-A6 if you can).")
    if SMOKE_TEST: print("SMOKE TEST: the numbers mean nothing. Set SMOKE_TEST = False for the real run.")

ALL DONE - no errors. Send Claude: the zip 'results_seed_*.zip' from the Output panel (and SS-A2 ... SS-A6 if you can).


## How to use this in the thesis
* Compare **S** (new seed) with the original model of the same kind (B or N): the difference shows how much a retrain alone moves each metric.
* After the four runs: report mean ± spread over the three seeds for B and for N, and the N − B difference for each seed. A gain that keeps its sign in all three seeds is much stronger evidence than one run.
* The post-processing of S is searched on its own OOF predictions, exactly as for B and N, so the whole pipeline is repeated.